## 상품 CRUD - products


In [ ]:
import os
from dotenv import load_dotenv
from supabase import create_client

# .env 파일의 환경변수 로드
load_dotenv()

url = os.environ.get("SUPABASE_URL")
key = os.environ.get("SUPABASE_ANON_KEY")

# supabase 클라이언트 객체 생성
supabase = create_client(url, key)

print(supabase)

In [ ]:
import os
from supabase import create_client

url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_ANON_KEY")

supabase = create_client(url, key)


class product_service:
    def __init__(self):
        data = supabase.auth.get_user()
        if data: self.user = data.user

    def create(self, name, price = 0, description = None):
        self.check_authority(True)

        response=(
            supabase.table("products")
            .insert({
                "name":name,
                "price":price,
                "description":description
                "seller_id": self.seller_id
            }).execute()
        )
        return response.data

    def update(self, id, data, name = None, price=None, description=None ):
        updated_data={}

        product = self.get_product(id)
        self.check_authority(user_id=product.user_id)
        
        if name is not None:
            update_data["name"] = name
        
        if price is not None:
            update_data["price"] = price

        if description is not None:
            update_data["description"] = description

        response=(
            supabase.table("products")
            .update(update_data)
            .execute()
        )
        return response.data


        #  수정 로직
        

    def delete(self, id):
        product = self.get_product(id)
        self.check_authority(user_id=product.user_id)

        response=(
            supabase.table("documents")
            .delete()
            .eq("id", id)
            .execute()
        )
        return response.data



    def get_product(self, id):
        response=(
            supabase.table("documents")
                    .select("*")
                    .eq("id", id)
                    .execute()
        )

        return response.data

    def get_products(self, page = 1, limit = 10, *, search = None):
        query = supabase.table("documents").select("*")

        if search:
            query = query.ilike("name",f"%{search}%")
        
        response =(
            .select()
            query
            .order("id")
            .execute()
        )

        return response.data

    def is_seller(self):
        return self.user and self.user.type == 'SELLER'

    def check_authority(self, is_create = False, '*', user_id = None):
        if not self.is_seller():
            raise PermissionError("판매자만 처리할 수 있습니다.")

        if (not is_created) and user_id != self.user.id:
            raise PermissionError("상품을 등록한 판매자만 수정 또는 삭제할 수 있습니다.")
